In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

variables = pd.read_parquet("data/clean/variables_modelo.parquet")
X = variables.drop(columns=["id_cliente", "sexo", "impago_mes_siguiente"])
y = variables["impago_mes_siguiente"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_entreno, X_valid, y_entreno, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
modelo_libre = joblib.load("data/clean/modelo_lightgbm.joblib")

In [2]:
direccion = {
    "retraso_sep": 1,
    "meses_con_retraso": 1,
    "peor_retraso_6m": 1,
    "limite_credito": -1,
    "parte_pagada_6m": -1,
}
restricciones = [direccion.get(col, 0) for col in X.columns]

for col, r in zip(X.columns, restricciones):
    if r != 0:
        print(f"{col:>20}: {'más → más riesgo' if r == 1 else 'más → menos riesgo'}")

         retraso_sep: más → más riesgo
   meses_con_retraso: más → más riesgo
     peor_retraso_6m: más → más riesgo
      limite_credito: más → menos riesgo
     parte_pagada_6m: más → menos riesgo


In [3]:
modelo_mono = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=7,
    max_depth=4,
    min_child_samples=300,
    reg_lambda=5.0,
    subsample=0.7,
    subsample_freq=1,
    colsample_bytree=0.7,
    monotone_constraints=restricciones,
    monotone_constraints_method="advanced",
    random_state=42,
    verbose=-1,
)
modelo_mono.fit(
    X_entreno, y_entreno,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)

def gini(modelo, X_, y_):
    return 2 * roc_auc_score(y_, modelo.predict_proba(X_)[:, 1]) - 1

print(f"Gini prueba · libre: {gini(modelo_libre, X_test, y_test):.3f} · "
      f"con restricciones: {gini(modelo_mono, X_test, y_test):.3f}")

Gini prueba · libre: 0.546 · con restricciones: 0.546


In [4]:
muestra = X_test.sample(1000, random_state=1)

def riesgo_medio(modelo, variable, valores):
    medias = []
    for v in valores:
        copia = muestra.copy()
        copia[variable] = v
        medias.append(modelo.predict_proba(copia)[:, 1].mean())
    return medias

valores = [-1, 0, 1, 2]
comprobacion = pd.DataFrame({
    "retraso_sep": valores,
    "PD media · libre": riesgo_medio(modelo_libre, "retraso_sep", valores),
    "PD media · con restricciones": riesgo_medio(modelo_mono, "retraso_sep", valores),
}).round(3)
display(comprobacion)

for nombre, modelo in [("libre", modelo_libre), ("con restricciones", modelo_mono)]:
    fallos = []
    for col, r in zip(X.columns, restricciones):
        if r == 0:
            continue
        rejilla = np.quantile(X_test[col], np.linspace(0.05, 0.95, 10))
        saltos = np.diff(riesgo_medio(modelo, col, rejilla)) * r
        if (saltos < -1e-6).any():
            fallos.append(col)
    print(f"Modelo {nombre}: variables que van en sentido contrario en algún tramo → {fallos or 'ninguna'}")

,retraso_sep,PD media · libre,PD media · con restricciones
0,-1,0.175,0.175
1,0,0.175,0.175
2,1,0.193,0.193
3,2,0.502,0.502


Modelo libre: variables que van en sentido contrario en algún tramo → ninguna


Modelo con restricciones: variables que van en sentido contrario en algún tramo → ninguna


In [5]:
joblib.dump(modelo_mono, "data/clean/modelo_lightgbm.joblib")
print("Guardado el modelo con restricciones")

Guardado el modelo con restricciones
